# LAB 3 — RAGAS + LangFuse Scores API (monitoração contínua)
## Aula 5 — Avaliação de Pipelines RAG

Instrumenta o pipeline RAG para avaliar **cada consulta individualmente** com RAGAS e enviar os scores ao **LangFuse Scores API** em tempo real, viabilizando *quality gate* em produção.

**Stack:**
- **LLM gerador & judge**: Groq `llama-3.1-8b-instant` (fallback Ollama `llama3.2:3b`)
- **Embeddings**: Ollama `bge-m3` (1024d)
- **Vector store**: OpenSearch 3.x — índice TCU 2026 da Aula 4
- **Observabilidade**: LangFuse Scores API + decorator `@observe`

**Pré-requisito:** LAB 1 e LAB 2 executados.

## 1. Instalação

In [ ]:
import subprocess, sys
for pkg in ['ragas>=0.1.16', 'datasets', 'pandas', 'matplotlib',
            'langchain>=0.2', 'langchain-core>=0.2',
            'langchain-groq>=0.1', 'langchain-ollama>=0.1',
            'opensearch-py>=2.7', 'python-dotenv>=1.0', 'langfuse>=2.36']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('✓ dependências instaladas')

## 2. Stack + LangFuse

In [ ]:
import os, json, time, warnings, datetime, statistics, math
from pathlib import Path
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

for env_path in [Path.cwd()/'.env', Path.home()/'mba-rag'/'.env',
                  Path.cwd().parent.parent/'.env']:
    if env_path.exists():
        load_dotenv(env_path, override=True); break

GROQ_API_KEY       = os.getenv('GROQ_API_KEY', '')
GROQ_LLM_MODEL     = os.getenv('GROQ_LLM_MODEL',   'llama-3.1-8b-instant')
OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL',    'http://localhost:11434')
OLLAMA_LLM_MODEL   = os.getenv('OLLAMA_LLM_MODEL',   'llama3.2:3b')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')
OPENSEARCH_HOST    = os.getenv('OPENSEARCH_HOST',     'localhost')
OPENSEARCH_PORT    = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER    = os.getenv('OPENSEARCH_USER',     'admin')
OPENSEARCH_PASS    = os.getenv('OPENSEARCH_PASSWORD', 'admin')
LANGFUSE_HOST      = os.getenv('LANGFUSE_HOST',       'http://localhost:3000')
LANGFUSE_PK        = os.getenv('LANGFUSE_PUBLIC_KEY', 'pk-lf-xxxx')
LANGFUSE_SK        = os.getenv('LANGFUSE_SECRET_KEY', 'sk-lf-xxxx')
INDEX_NAME         = os.getenv('INDEX_NAME', 'corpus_juridico_aula4')

META = {'faithfulness': 0.80, 'answer_relevancy': 0.75,
        'context_recall': 0.70, 'context_precision': 0.70}

from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from opensearchpy import OpenSearch
from ragas.llms       import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langfuse import Langfuse

# LLM (Groq primário, Ollama fallback)
llm = None; LLM_PROVIDER = None; LLM_MODEL = None
if GROQ_API_KEY:
    try:
        c = ChatGroq(model=GROQ_LLM_MODEL, api_key=GROQ_API_KEY,
                     temperature=0.0, max_tokens=512, timeout=30)
        c.invoke([HumanMessage(content='ok')])
        llm, LLM_PROVIDER, LLM_MODEL = c, 'groq', GROQ_LLM_MODEL
    except Exception:
        pass
if llm is None:
    llm = ChatOllama(model=OLLAMA_LLM_MODEL, base_url=OLLAMA_BASE_URL,
                     temperature=0.0, num_predict=512)
    LLM_PROVIDER, LLM_MODEL = 'ollama', OLLAMA_LLM_MODEL

embeddings = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)
_ = embeddings.embed_query('teste'); assert len(_) == 1024

os_client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False, verify_certs=False, timeout=60,
)

try:
    lf = Langfuse(public_key=LANGFUSE_PK, secret_key=LANGFUSE_SK, host=LANGFUSE_HOST)
    lf.auth_check(); LANGFUSE_OK = True
    print(f'✓ LangFuse {LANGFUSE_HOST}')
except Exception as e:
    lf = None; LANGFUSE_OK = False
    print(f'⚠ LangFuse indisponível ({e}) — modo degradado')

ragas_llm        = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

print(f'Stack ✓ | LLM={LLM_PROVIDER}/{LLM_MODEL} | embed={OLLAMA_EMBED_MODEL} | OS={os_client.info()["version"]["number"]} | index={INDEX_NAME}')

## 3. Classe `RAGPipelineComAvaliacao`

Encapsula: retrieval (kNN BGE-M3) → geração (Groq/Ollama) → avaliação RAGAS → envio LangFuse Scores API.

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
import pandas as pd

PROMPT_GEN = ChatPromptTemplate.from_messages([
    ('system', 'Você é um assistente jurídico brasileiro. Responda APENAS com base nos trechos.'),
    ('human', 'TRECHOS:\n{contextos}\n\nPERGUNTA: {pergunta}\n\nRESPOSTA:')
])
GEN_CHAIN = PROMPT_GEN | llm | StrOutputParser()
METRICAS = [faithfulness, answer_relevancy, context_recall, context_precision]

class RAGPipelineComAvaliacao:
    def __init__(self, os_client, embeddings, lf, indice: str, k: int = 5):
        self.os_client = os_client
        self.embeddings = embeddings
        self.lf = lf
        self.indice = indice
        self.k = k

    def _recuperar(self, q: str) -> list[str]:
        try:
            v = self.embeddings.embed_query(q)
            r = self.os_client.search(
                index=self.indice,
                body={'size': self.k,
                      'query': {'knn': {'embedding': {'vector': v, 'k': self.k}}},
                      '_source': ['conteudo', 'titulo']},
            )
            ctxs = []
            for h in r['hits']['hits']:
                s = h['_source']
                txt = (s.get('titulo','') + '\n' + s.get('conteudo','')).strip()
                if txt: ctxs.append(txt)
            return ctxs or ['Nenhum contexto.']
        except Exception as e:
            return [f'Erro retrieval: {e}']

    def _gerar(self, q: str, ctxs: list[str]) -> str:
        bloco = '\n\n'.join(f'[Doc {i+1}]\n{c[:600]}' for i, c in enumerate(ctxs))
        try:
            return GEN_CHAIN.invoke({'contextos': bloco, 'pergunta': q}).strip()
        except Exception as e:
            return f'[Erro geração: {e}]'

    def _avaliar(self, q, a, ctxs, gt) -> dict:
        ds1 = Dataset.from_dict({'question':[q], 'answer':[a],
                                  'contexts':[ctxs], 'ground_truth':[gt]})
        try:
            r = evaluate(ds1, metrics=METRICAS, llm=ragas_llm,
                         embeddings=ragas_embeddings, raise_exceptions=False)
            return {m.name: float(r[m.name]) if not math.isnan(float(r[m.name])) else 0.0
                    for m in METRICAS}
        except Exception as e:
            print(f'  ⚠ RAGAS erro: {e}')
            return {m.name: 0.0 for m in METRICAS}

    def run(self, question: str, ground_truth: str) -> dict:
        t0 = time.time()
        trace = None; trace_id = None
        if self.lf:
            try:
                trace = self.lf.trace(
                    name='RAG-Pipeline-Juridico',
                    input={'question': question, 'ground_truth': ground_truth[:120]},
                    metadata={'indice': self.indice, 'k': self.k,
                               'llm_provider': LLM_PROVIDER, 'llm_model': LLM_MODEL,
                               'embed_model': OLLAMA_EMBED_MODEL, 'lab': 'LAB3'},
                    tags=['rag', 'juridico', 'lab3', 'ragas-integrado'],
                )
                trace_id = trace.id
            except Exception as e:
                print(f'  ⚠ trace err: {e}')

        # retrieval span
        t1 = time.time(); ctxs = self._recuperar(question); t2 = time.time()
        if trace:
            try:
                trace.span(name='retrieval-knn-bge-m3',
                            input={'question': question, 'k': self.k},
                            output={'n_contextos': len(ctxs),
                                    'preview': ctxs[0][:120] if ctxs else ''},
                            metadata={'latencia_ms': round((t2-t1)*1000, 1),
                                      'embed_model': OLLAMA_EMBED_MODEL},
                            start_time=datetime.datetime.fromtimestamp(t1),
                            end_time=datetime.datetime.fromtimestamp(t2))
            except Exception: pass

        # geração span
        t3 = time.time(); ans = self._gerar(question, ctxs); t4 = time.time()
        if trace:
            try:
                trace.span(name=f'geracao-{LLM_PROVIDER}',
                            input={'question': question, 'n_contextos': len(ctxs)},
                            output={'answer': ans[:300]},
                            metadata={'latencia_ms': round((t4-t3)*1000, 1),
                                      'llm_provider': LLM_PROVIDER, 'llm_model': LLM_MODEL},
                            start_time=datetime.datetime.fromtimestamp(t3),
                            end_time=datetime.datetime.fromtimestamp(t4))
            except Exception: pass

        # avaliação RAGAS
        scores = self._avaliar(question, ans, ctxs, ground_truth)

        # envio scores → LangFuse Scores API
        if self.lf and trace_id:
            for n, v in scores.items():
                try:
                    meta = META.get(n, 0)
                    self.lf.score(trace_id=trace_id, name=n, value=v,
                                   comment=f'meta={meta} | Δ={v-meta:+.4f}')
                except Exception as e:
                    print(f'  ⚠ score {n}: {e}')
            try:
                trace.update(output={'answer': ans[:300], 'scores': scores,
                                      'tempo_total_s': round(time.time()-t0, 2)})
                self.lf.flush()
            except Exception: pass

        return {'question': question, 'answer': ans, 'contexts': ctxs,
                'ground_truth': ground_truth, 'scores': scores,
                'trace_id': trace_id, 'tempo_s': round(time.time()-t0, 2)}

    def run_batch(self, pares: list, max_pares: int = 10) -> list:
        out = []
        print(f"{'#':<4}{'Faith':>8}{'AnsRel':>8}{'CtxRec':>8}{'CtxPre':>8}{'trace_id':<36}{'t(s)':>7}")
        print('-' * 80)
        for i, p in enumerate(pares[:max_pares], 1):
            r = self.run(p['question'], p['ground_truth'])
            out.append(r); s = r['scores']
            print(f"{i:<4}{s['faithfulness']:>8.4f}{s['answer_relevancy']:>8.4f}"
                   f"{s['context_recall']:>8.4f}{s['context_precision']:>8.4f}"
                   f"{str(r['trace_id'] or 'N/A')[:34]:<36}{r['tempo_s']:>7.1f}")
        return out

pipeline = RAGPipelineComAvaliacao(os_client, embeddings, lf, INDEX_NAME, k=5)
print('✓ pipeline pronto')

## 4. Executar Batch (10 pares) com Envio Automático ao LangFuse

In [ ]:
DATASET_LAB1 = 'dataset_avaliacao_completo.json'
if not os.path.exists(DATASET_LAB1):
    raise FileNotFoundError(f'{DATASET_LAB1} não encontrado. Execute o LAB 1 primeiro.')
with open(DATASET_LAB1, encoding='utf-8') as f:
    dataset = json.load(f)
print(f'✓ {len(dataset)} pares carregados — processando 10 para o batch')

resultados_batch = pipeline.run_batch(dataset, max_pares=10)
print(f'\n✓ batch concluído: {len(resultados_batch)} pares processados')

## 5. Verificar Traces no LangFuse

In [ ]:
print('=== Traces registrados no LangFuse ===')
for i, r in enumerate(resultados_batch, 1):
    tid = r['trace_id']
    if tid:
        print(f'  {i:02d} → {LANGFUSE_HOST}/traces/{tid}')
    else:
        print(f'  {i:02d} → (sem trace; LangFuse offline)')
print(f'\nDashboard: {LANGFUSE_HOST}/dashboard')

## 6. Dashboard de Qualidade — Médias vs Metas

In [ ]:
import matplotlib.pyplot as plt, numpy as np

medias  = {m: statistics.mean([r['scores'][m] for r in resultados_batch]) for m in META}
desvios = {m: (statistics.stdev([r['scores'][m] for r in resultados_batch])
                if len(resultados_batch) > 1 else 0.0) for m in META}

print('=== Dashboard de Qualidade (10 pares) ===')
print(f"{'métrica':<22}{'média':>8}{'desvio':>8}{'meta':>8}{'status':>10}{'Δ':>9}")
print('-' * 65)
for m, target in META.items():
    med = medias[m]; dv = desvios[m]
    st  = '✓' if med >= target else '⚠'
    print(f'{m:<22}{med:>8.4f}{dv:>8.4f}{target:>8.2f}{st:>10}{med-target:>+9.4f}')

nomes = ['Faithfulness','AnsRel','CtxRec','CtxPre']
vals  = [medias[m] for m in META]; metas = [META[m] for m in META]
errs  = [desvios[m] for m in META]
cores = ['#2ecc71' if v >= t else '#e74c3c' for v, t in zip(vals, metas)]
x = np.arange(len(nomes))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(x, vals, color=cores, alpha=0.85, yerr=errs, capsize=5,
              error_kw={'ecolor': '#555', 'lw': 1.5})
for i, t in enumerate(metas):
    ax.hlines(t, i-0.35, i+0.35, colors='red', linestyles='--', linewidths=2)
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02,
             f'{v:.3f}', ha='center', fontweight='bold')
ax.set_ylim(0, 1.15); ax.set_xticks(x); ax.set_xticklabels(nomes)
ax.set_ylabel('Score (0–1)'); ax.grid(axis='y', alpha=0.4, linestyle=':')
ax.set_title(f'RAGAS — LAB3 (LLM={LLM_PROVIDER}/{LLM_MODEL})', fontweight='bold')
plt.tight_layout(); plt.savefig('dashboard_qualidade_lab3.png', dpi=140); plt.show()
print('✓ dashboard_qualidade_lab3.png salvo')

## 7. Padrão `@observe` para Uso em Produção

O decorator `@observe` do LangFuse cria/fecha o trace automaticamente — sem boilerplate.

In [ ]:
try:
    from langfuse.decorators import observe, langfuse_context
    OBS_OK = True
except Exception:
    OBS_OK = False
    print('@observe indisponível nesta versão do langfuse — pule esta célula')

if OBS_OK:
    @observe(name='rag-juridico-com-ragas')
    def rag_producao(question: str, ground_truth: str) -> dict:
        langfuse_context.update_current_observation(
            metadata={'dominio': 'juridico', 'versao': '1.0', 'lab': 'LAB3'},
            tags=['producao', 'rag', 'ragas-integrado'])
        ctxs   = pipeline._recuperar(question)
        ans    = pipeline._gerar(question, ctxs)
        scores = pipeline._avaliar(question, ans, ctxs, ground_truth)
        tid = langfuse_context.get_current_trace_id()
        if lf and tid:
            for n, v in scores.items():
                lf.score(trace_id=tid, name=n, value=v,
                          comment=f'@observe | meta={META.get(n,0)}')
        return {'answer': ans, 'scores': scores, 'trace_id': tid}

    demo = rag_producao(dataset[0]['question'], dataset[0]['ground_truth'])
    print(f'\ntrace_id (@observe): {demo["trace_id"]}')
    print(f'scores: {demo["scores"]}')

## 8. Exportar Resultados

In [ ]:
linhas = []
for i, r in enumerate(resultados_batch, 1):
    s = r['scores']
    linhas.append({
        'par': i, 'question': r['question'][:160],
        'answer': r['answer'][:160], 'ground_truth': r['ground_truth'][:160],
        'n_contextos': len(r['contexts']), 'trace_id': r['trace_id'] or '',
        'tempo_s': r['tempo_s'],
        **{m: s[m] for m in META},
        **{f'{m}_ok': s[m] >= META[m] for m in META},
        'langfuse_url': f'{LANGFUSE_HOST}/traces/{r["trace_id"]}' if r['trace_id'] else ''
    })
df = pd.DataFrame(linhas)
OUT_CSV = 'ragas_langfuse_integrado_resultados.csv'
df.to_csv(OUT_CSV, index=False, encoding='utf-8')
print(f'✓ {OUT_CSV} ({os.path.getsize(OUT_CSV)/1024:.1f} KB)')

cols_ok = [f'{m}_ok' for m in META]
df['todas_ok'] = df[cols_ok].all(axis=1)
print(f'\nPares com TODAS as metas: {df["todas_ok"].sum()}/{len(df)}')
for c in cols_ok:
    print(f'  {c}: {df[c].sum()}/{len(df)}')

## Referências

ES, S. et al. **RAGAS**. arXiv:2309.15217, 2023.  
LANGFUSE. **Scores API + @observe decorator**. <https://langfuse.com/docs/scores>.  
GROQ INC. **Groq API**. <https://console.groq.com/docs>.